# Functions

One of the greatest ability of the human mind is the ability to abstract - and what could be a better concept of abstraction then the concept of functions? Functions help us compact infinite information into a finite space. 

For example, consider the very loved function, $F(x) = x^2$. If we consider only positive integer values for x, the information contained in this function is equivalent to the following table:

| Value of x | Value of F(x) |
| ---------- | ------------- |
| 1 | 1 |
| 2 | 4 |
| 3 | 9 |
| 4 | 16 | 
| ... | ... |

... we can keep filling this table all the way to $+\infty$. This function, **F(x)**, essentially converts this infinite length table in a simple one line statement: $F(x) = x^2$.

This is is the power of functons. Functions are able to abstract out a very large (or even $+\infty$) **state spaces** to a finite and consise form. 

This gives us the motivation to develop function-based Learning Methods where we replace the **Q Table** (which is a table with state-action pairs) to a **Q Function** - which is a function that aims to compact the Table of Q into a simple and concise statement form.

## Baby Steps

First lets consider our favourite, Monte Carlo Update Algorithm!

The Update equation is:
$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big( G_t - Q(s, a) \Big)$$


Where Q(s,a) is a Q Table which may like:

| State s | Action a | Value of Q(s, a) |
| ---------- | ------------- | --- |
| 1 | 1 | 0.4 | 
| 1 | 2 | - 0.2 | 
| 1 | 3 | 4 |
| 1 | 4 | 9 |
| 2 | 1 | 67 |
| 2 | 2 | 2 |
| 2 | 3 | 4 |
| 2 | 4 |  5 |
| ... | ... | ...|

Obviously the values of Q(s,a) is arbitrary in this table. But the size of this table will be | S | * | A | where S is the set of states and A is the Set of Actions. For cliff walking, this table will have a size of ( 37 ) * ( 4 ) = 148 distinct rows.

Imgine, instead of having 37 states, we had 10,000 states and 200 actions. We would have 2,000,000 distinct rows whose Q(s, a) we will have to "learn" by interacting with the environment. Firstly, that would take a long time to update accurate values, and, in some cases might even be impossible to learn in one human lifetime (80 years). Another issue is the issue of constantly keeping this much information in memory - 2,000,000 distict rows is no joke to store.

This is where we can use functions to *approximate* the values of Q(s, a) instead of storing the entire table, we will only store the function and calculate the actual Q-Values in runtime!

So, consider the following Q Function:

$$Q_{function} (w, a, s) = F(w, a, s) $$

Where w is a vector (which is a learnable parameter), and assume F(w, a, s) gives us the correct value of taking action, a, at state s for any valid values of a and s.

The our policy can be rewritten from:

$$
\pi_{*} = \argmax_{a} (Q_{table}(a, s));


\forall a \in A, s \in S
$$

to

$$
\pi_{*} = \argmax_{a} (Q_{function}(w, a, s));
\forall a \in A, s \in S
$$ 

when Q-Table and $Q_{function} are optimal.

However, our problem then becomes how can I update the value of w s.t F(w, a, s) will always output the correct Q(s, a) Value. Moreover, how can we decide how F(w, a, s) will look like?



### How does F(w, a, s) look like?

The straightforward answer is - *we don't know*. There is probably no universal function, F(w, a, s), that will work for solving all problems. But we can take guess a function that can work and check if it is able to correctly estimate values of Q(s, a). Lets look at some candidate functions for our table:

#### Linear Function

$F(w, a, s) = w_{1} * s + w_{2} * s + w_{3} * s + w_{4} * s + a$

Where $w_k$ is the $k-th$ element of vector $w$

For a 4 dimentional vector w. Of course, we can use more dimentions for vector w and see if it can work. 

---

#### Power Function
$ F(w, a, s) = w_{1} * s + w_{2} * s^2 + w_{3} * s^3 + w_{4} * s^4 + a$

For a 4 dimentional vector w. Of course, we can use more dimentions for vector w and see if it can work.

---


#### Core Properties

The core properties for the candidate function F(w, a, s) is that it can:

1. Domain of s, is greater than or requal to the set, S, which contains all values of s. (Trivial)
2. Domain of a, is greater than or requal to the set, A, which contains all values of a. (Trivial)
3. Correctly able to map state-action space to value space (Difficult)

The third point is the one that is diffult to satisfy. We have to perform trial and error to really know which function works.


## Optimise F

Once we have a candidate Fuction, we need to optimise w. We want to reduce the error:

$$ min_{w}(G_t - F(w, s, a)) $$

Where $G_t$ is the actual return we get at state s, taking action a and following the our policy $\pi$ until termination. However, instead of minsing $ (G_t - F(w, s, a)) $, we should mininmise:

$$ min_{w} [G_t - F(w, s, a)]^2 $$

Because it will penalise greater deviations more. Furthermore, we are not trying to minimise our Error at just state s and a, but for all states and all actions. We can write all that down succintly by saying:

$$ min_{w} \sum_{a , s} [G_t - F(w, s, a)]^2 ; \forall a \in A, s \in S  $$


### Minimise

Now let's actually perform minization. Easy way to do this, is simply differentiating F and then following **down-slope**, of course, given F is differentiable. Which would mean:

$$ w \leftarrow w - \frac{\partial}{\partial w} \sum_{a , s} [G_t - F(w, s, a)]^2 $$

Note that if we want to go above slope, we would use a **plus** (+) sign instead of **minus** (-)

Now lets incorporate a learning rate, call it $\alpha$ and introduce a constant scaling of $\frac{1}{2}$. Our equation will become:

$$w \leftarrow w - \alpha \frac{1}{2} \frac{\partial}{\partial w} \sum_{a , s} [G_t - F(w, s, a)]^2 $$

Lets simplify by performing the differntiation:

$$   \alpha \frac{1}{2} \frac{\partial}{\partial w} \sum_{a , s} [G_t - F(w, s, a)]^2 $$
$$ =  \alpha \frac{1}{2} \sum_{a , s} 2 * [G_t - F(w, s, a)] * (-1) \frac{\partial}{\partial w} F(w, s, a) $$
$$ = \alpha \frac{1}{2} * 2 * (-1) \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)]$$
$$ =  - \alpha \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)]$$


Substituting the above differentiation to our optimization equation, it becomes:

$$ w \leftarrow w - ( - \alpha \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)] )$$

or, simply

$$ w \leftarrow w + \alpha \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)]$$



### Limitations

Notice that we now have limitations with $ F(w, s, a)$ since we have to calculate $\frac{\partial}{\partial w} F(w, s, a)$ which means our q-function, F, has to be differentiable. Another limitation is that we need the complete return, $G_t$, of an episode before we start making updates to our q-function. So our Agent will be acting completely randomly on the first episode and if the episodes are very long, our agent will take a long time to converge to optimal policy. 

Keeping these limitations in mind, let's see an example on how we can implement a functional approach to RL instead of the traditional q tables.

### Implementation

Instead of worrying about whether our function will be able to effectively model the environment, lets use a fool proof function to do it! Note that for cliff walking there are less than 48 states (4*12 = 48).

Define:
$$ F(w,s,a) = w_a * x(s) $$

s.t 

$$ \frac{\partial}{\partial w} F(w,s,a) =  x(s) $$

Where:
- $w_a$ is a 48-dimensional vector depending on value of a. so if we have $w_0$, $w_1$, $w_2$ and $w_3$
- x(s) is a 48-dimensional vector that depends on s. We define x(s) as:

$$ x_k = \begin{cases} \text{1} &  \text{if k = s } \\ 
\text{0} &  \text{if k } {\neq s } \end{cases} 
\quad \text{; where } x_k \text{ is the } k^{th} \text{term in } x(s) 
$$

in code it may look like:

```python

def x(s):
    temp = np.zeros(48) # 48 columns
    temp[s] = 1
    return temp

w = np.zeros((4, 48)) # 48 cols, 4 rows

def F(w, s, a):
    weights = w[a]
    result = np.dot(weights, x(s))

    return result


def dwF(w, s, a):
    return x(s)
```

This is a very popular candidate Q-Function and is known as Linear Value Function Approximation with one-hot state encoding.

### Implementation

Lets actually implement this in python!

In [ ]:
from utils import *
import numpy as np
import gymnasium as gym

epsilon = 0.1
epsilon_decay_rate = 0.999
epsilon_min = 0.01

gemma = 0.9
alpha = 0.1
num_episodes = 5000
MAX_STEPS_PER_EPISODE = 500

seed = 42

env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)

action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)

weights = np.zeros((len(action_space), len(observation_space)))

def x(state):
    temp = np.zeros(len(observation_space))
    temp[state] = 1
    return temp

def q_function(weights, state, action):
    weight_vector = weights[action]
    value = np.dot(weight_vector, x(state=state))
    return value

def q_function_differential(weights, state, action):
    return x(state=state)


episode_lengths = []
episode_rewards = []
episode_durations = []
episode_avg_mc_error = []

In [ ]:
rng = np.random.default_rng(seed=seed)


def get_best_acton(weights, state):
    num_actions = weights.shape[0]
    best_action_idx = 0
    best_action_val = weights[0, state]
    for action in range(num_actions):
        if best_action_val < weights[action, state]:
            best_action_idx = action
            best_action_val = weights[action, state]

    return best_action_idx


for episode in range(num_episodes):

    start_time = datetime.now()
    env.reset()

    # Array to store, State, Action, Reward for the current episode. WE use it to backtrack and update the Q-values at the end of the episode.
    episode_data = []
    terminated = False
    truncated = False

    state, info = env.reset()
    epsilon = max(epsilon * epsilon_decay_rate, epsilon_min)  # Decay epsilon after each episode

    while not (terminated or truncated):

        # Epsilon-greedy action selection
        if rng.random() < epsilon:
            action = rng.choice(action_space)
        else:
            action = get_best_acton(weights=weights, state=state)

        next_state, reward, terminated, truncated, info = env.step(action)

        episode_data.append((state, action, reward))

        state = next_state

    # Calculate returns and update Q-Function weights
    G = 0
    for state, action, reward in reversed(episode_data):
        G = gemma * G + reward
        TD_Error = G - q_function(weights=weights, state=state, action=action)
        weights[action] += alpha * TD_Error * q_function_differential(weights=weights, state=state, action=action)


    # Average episode error for monitoring convergence
    Avg_MC_Episode_Error = np.mean([np.abs(G - q_function(weights=weights, state=state, action=action)) for state, action, reward in episode_data])
    episode_avg_mc_error.append(Avg_MC_Episode_Error)


    end_time = datetime.now()
    episode_duration = (end_time - start_time).total_seconds()
    episode_durations.append(episode_duration)
    episode_rewards.append(sum([reward for _, _, reward in episode_data]))
    episode_lengths.append(len(episode_data))


    if (episode + 1) % 200 == 0:
        print(f"Episode: {episode + 1}, Average MC Episode Error: {Avg_MC_Episode_Error:.4f}",
                f"Episode Duration: {episode_duration:.4f} seconds",
                f"Total Reward: {episode_rewards[-1]}, Episode Length: {episode_lengths[-1]}",
                f"Epsilon: {epsilon:.4f}")

## Monte Carlo Neural Network Approximation

Lets use a Neural Network for a functional approximation. Here are some things to note:

- **State** : Instead of inputing a the raw state number, we will transform a state to a one-hot encoding array. For example, state 1 will be transformed in to [0,1,0,0,0,0...] and State 2 will be transformed into [0,0,1,0,0,0...] etc. Bascially instead of represeting states as *unique numbers*, we will represent it as *unique arrays*. The one-hot encoded array size will always be the length of the observation space, in our case, 48. We perform one hot encoding instead of passing in the state as an integer because 

- **Output** : Our Neural Network function will output an array of 4 numbers for every state input, represeting the 4 distict action values (expected utility for going up, right, down left) for the input state.

- **Structure** :  We Will use one input layer (1x48), one hidden layer of 64 neurons and one output layer. The choice is arbitrary, however, it is to be noted that sometimes some structures will not yeild good results.

<p align="center">
<img src="images/qNetwork_Structure.png" h="100px">
</p>

In [ ]:
import torch
from torch import nn, optim
from utils import *
import numpy as np
import gymnasium as gym
import os

import warnings
warnings.filterwarnings('ignore')


# ----------------------------------- #
#  Hyper-parameters
# ----------------------------------- #
gamma = 0.9
epsilon = 0.5
epsilon_decay_rate = 0.9995 
epsilon_min = 0.05
alpha = 0.001                
num_episodes = 2000


# ----------------------------------- #
#  🏝️ Environment Setup
# ----------------------------------- #
seed = 42
rng = np.random.default_rng(seed=seed)
MAX_STEPS_PER_EPISODE = 500
env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)
action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)


# ----------------------------------- #
#  📐 Neural Network Architecture
# ----------------------------------- #
class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()

        self.liner_relu_stacks = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LeakyReLU(),
            nn.Linear(64, output_dim)        
            )
    
    def forward(self, x):
        return self.liner_relu_stacks(x)
    

device = torch.device("cpu")

qNetwork = QNetwork(len(observation_space), len(action_space)).to(device)
optimizer = optim.Adam(qNetwork.parameters(), lr=alpha)
loss_fn = nn.MSELoss()

state, info = env.reset()

episode_lengths = []
episode_rewards = []
episode_durations = []
episode_avg_mc_error = []

weights_path = "weights/q_network_weights.pth"
training_data_path = "data/train_data.json"



# ----------------------------------- #
#  👷 Helper Methods
# ----------------------------------- #
def get_state_tensor(state, observation_space):
    one_hot = np.zeros(len(observation_space))
    one_hot[state] = 1
    return torch.tensor(one_hot).unsqueeze(0).float().to(device)

def choose_action(state, epsilon):
    if rng.random() < epsilon:
        return rng.choice(action_space)
    else:
        with torch.no_grad():
            state_tensor = get_state_tensor(state, observation_space)
            q_values = qNetwork(state_tensor)
            return torch.argmax(q_values).item()
        
def generate_qtable(qNetwork, observation_space):
    q_table = np.zeros((len(observation_space), len(action_space)))

    for idx in range(len(q_table)):
        with torch.no_grad():
                state_tensor = get_state_tensor(state=idx, observation_space=observation_space)
                predicted_vals = qNetwork(state_tensor)
                q_table[idx] = np.array(predicted_vals[0], copy=True, dtype=float)
    return q_table


# ----------------------------------- #
#  🔄 Save & Load Checkpoints 
# ----------------------------------- #
def load_checkpoint(model, optimizer, file_path):
    """Loads a saved training checkpoint to resume model optimization.

    Checks for the existence of a binary checkpoint file at the specified path. 
    If found, it reconstructs the model parameters (weights/biases) and restores 
    the optimizer state (momentum buffers) to guarantee training continuation 
    stability. It also retrieves the corresponding historical exploration 
    rate (epsilon) and episode counters.

    Args:
        model (torch.nn.Module): The active neural network instance to inject 
            the saved weights into.
        optimizer (torch.optim.Optimizer): The active optimizer tracking historical 
            gradients/momentum parameters to restore.
        file_path (str): The local disk target path to the `.pth` binary checkpoint.

    Returns:
        tuple (float, int): A pair containing:
            - epsilon (float): The current exploration probability rate. 
              Defaults to 1.0 if not found in the checkpoint.
            - episode (int): The last completed episode index. 
              Defaults to 0 if starting fresh.
    """
    if os.path.exists(file_path):
        print(f"🔄 Found existing checkpoint '{file_path}'. Loading weights...")
        checkpoint = torch.load(file_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        return checkpoint.get('epsilon', 1.0), checkpoint.get('episode', 0)
    else:
        print(f"⚠️ No checkpoint found at '{file_path}'. Starting from scratch.")
        return 1.0, 0

def save_checkpoint(model, optimizer, epsilon, episode, file_path):
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epsilon': epsilon,
        'episode': episode
    }, file_path)

def delete_checkpoint(file_path):
    ## Delete weights and data
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"🗑️ Weight file '{file_path}' has been deleted from your disk.")


# ⚠️
# Uncomment this if you want to resume training from previous attempt
# epsilon, episode = load_checkpoint(model=qNetwork,
#                 optimizer=optimizer,
#                 file_path=weights_path)


q_table = generate_qtable(qNetwork=qNetwork, observation_space=observation_space)
plot_q_table(q_table=q_table)



# ----------------------------------- #
#  🔁 Training Loop
# ----------------------------------- #
for episode in range(num_episodes):
    start_time = datetime.now()
    env.reset()

    episode_data = []
    episode_reward = 0
    terminated = False
    truncated = False

    state, info = env.reset()
    epsilon = max(epsilon * epsilon_decay_rate, epsilon_min)


    while not (terminated or truncated):
        action = choose_action(state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        
        episode_data.append((state, action, reward))
        state = next_state
        

    # Calculate returns and update Q-Function weights
    G = 0
    absolute_errors = []
    optimizer.zero_grad()
    for state, action, reward in reversed(episode_data):
        G = reward + gamma * G 

        state_tensor = get_state_tensor(state=state, observation_space=observation_space)
        assert state_tensor[0][state].item() == 1, "Assert correct state tensor"

        q_values = qNetwork(state_tensor)
        
        predicted_q = q_values[0, action]
        target_tensor = torch.tensor([G], dtype=torch.float32).to(device)
        
        loss = loss_fn(predicted_q.unsqueeze(0), target_tensor)
        with torch.no_grad():
            expected_loss_val = (G - predicted_q.item()) ** 2
            expected_tensor = torch.tensor([expected_loss_val], dtype=torch.float32).to(device)
            
            assert torch.allclose(loss, expected_tensor, atol=1e-5), \
                f"Loss mismatch! Manual: {expected_loss_val:.6f}, PyTorch: {loss.item():.6f}"
                    
    
        # ---------------------- Perform the step and update weights. ---------------------- #
        loss.backward()

        # Scale the gradients
        torch.nn.utils.clip_grad_norm_(qNetwork.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        
        with torch.no_grad():
            absolute_errors.append(torch.abs(target_tensor - predicted_q).item())
        # --------------------------------------------------------------------------------- #


    Avg_Episode_Error = np.mean(absolute_errors)
    episode_avg_mc_error.append(Avg_Episode_Error)

    end_time = datetime.now()
    episode_duration = (end_time - start_time).total_seconds()
    episode_durations.append(episode_duration)
    episode_rewards.append(sum([reward for _, _, reward in episode_data]))
    episode_lengths.append(len(episode_data))

    if (episode + 1) % 100 == 0:
        avg_rolling_reward = np.mean(episode_rewards[-100:])
        print(f"Episode: {episode + 1}, Average MC Episode Error: {Avg_Episode_Error:.4f}",
                f"Episode Duration: {episode_duration:.4f} seconds,",
                f"Average Reward: {avg_rolling_reward:.2f}, Latest Reward: {episode_rewards[-1]} Episode Length: {episode_lengths[-1]} | "
                f"Epsilon: {epsilon:.4f}")
        
        save_checkpoint(model=qNetwork, optimizer=optimizer, epsilon=epsilon, episode=episode, file_path=weights_path)

    if (episode + 1) % 400 == 0:
        q_table = generate_qtable(qNetwork, observation_space=observation_space)
        plot_q_table(q_table=q_table)


#### Comments


- **Learning rate ($\alpha$)**: The Learning Rate has to be lower the previous tabular methods since the minimization we are trying to run on the multi-dimensional hyperplane created from the neurons are quite sensitive. This statement is based on our observation. We will set is between $1^{-3}$ to $1^{-4}$.

- `SGD Optimizer` seems to descend to the optimum too slowly - requiring far more than 2000 episodes and slower epsilon decay (or a different epsilon strategy entirely). However, `Adam Optimizer` with the power of varying learning rates for higher slopes learn much faster and weights descents to local optima in less than 2000 episodes.

- **Scaled Gradient**: The function `torch.nn.utils.clip_grad_norm_` is applied to the gradients of our parameters to scale the gradient values down if they are too large. This is done so that our agent learns gradually even when the gradients are very large - as a precaution to not over-step in the hyperplane. This leads to much more stable learning. Note this clipping preserves the direction of the "step" and only changes the magnitude.

Scaling Factor Formula:

$$
\frac{\text{max norm}}{\sqrt{\sum grad^2 }}
$$


## Double Q-Learning with Neural Networks (DQN)

The Key Idea is to use **Two Q-Networks**: Target Network and Policy Network. Sample action from `Policy Network` and adjust parameters of `Target Network` from the data collected. Update the `Policy Network` parameters by syncing to the parameters of the `Target Network` occasionally. This Helps us perform approximations and give our agent "intelligence" from its experience even before an episode ends. 


Notes:
- **epsilon decay**: We can use a higher epsilon decay and reduce our epsilon faster since DQN method performs learning loops multiple times in a single episode and thus is expected to learn much faster. So we can afford to reduce our epsilon faster so that our agent behaves optimally earlier. The previous issue of slow learning is not a factor for this method.

In [ ]:
import torch
from torch import nn, optim
from utils import *
import numpy as np
import gymnasium as gym
import os

epsilon = 0.5
epsilon_decay_rate = 0.9999
epsilon_min = 0.01

gamma = 0.9
alpha = 0.001
num_episodes = 5000
MAX_STEPS_PER_EPISODE = 500

seed = 42
rng = np.random.default_rng(seed=seed)

env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)

action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)


input_dim = len(observation_space)
output_dim = len(action_space)

device = torch.device("cpu")

print(f"Using device: {device}")

class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()

        self.liner_relu_stacks = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.Linear(64, output_dim)        
            )
    
    def forward(self, x):
        return self.liner_relu_stacks(x)
        

qNetwork = QNetwork(input_dim=input_dim, output_dim=output_dim)
qNetwork = qNetwork.to(device)
optimizer = optim.SGD(qNetwork.parameters(), lr=alpha)
loss_fn = nn.MSELoss()

state, info = env.reset()

episode_lengths = []
episode_rewards = []
episode_durations = []
episode_avg_mc_error = []


weights_path = "q_network_weights.pth"
training_data_path = "train_data.json"
isDelete = False
## Delete weights and data
if os.path.exists(weights_path) and isDelete:
    os.remove(weights_path)
    print(f"🗑️ Weight file '{weights_path}' has been deleted from your disk.")

if os.path.exists(training_data_path) and isDelete:
    os.remove(training_data_path)
    print(f"🗑️ Weight file '{training_data_path}' has been deleted from your disk.")


def get_state_tensor(state, observation_space):
        one_hot = np.zeros(len(observation_space))
        one_hot[state] = 1
        state_tensor = torch.tensor(one_hot).unsqueeze(0).float().to(device)
        return state_tensor


def get_best_acton(qNetwork, state, observation_space):
    with torch.no_grad():
        state_input = get_state_tensor(state, observation_space=observation_space)
        logits = qNetwork(state_input)
        return torch.argmax(logits).item()

def choose_action(state, action_space, observation_space, epsilon):
    if rng.random() < epsilon:
        action = rng.choice(action_space)
    else:
        action = get_best_acton(state=state, qNetwork=qNetwork, observation_space=observation_space)
    return action


    

In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import deque
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')


# --- Hyperparameters ---
gamma = 0.9
epsilon = 0.5
epsilon_decay_rate = 0.99 
epsilon_min = 0.05
alpha = 0.001                
num_episodes = 2000
batch_size = 64              # Experience batch size
target_update_frequency = 10 # Sync after 10 episodes

device = torch.device("cpu")


MAX_STEPS_PER_EPISODE = 500

seed = 42
rng = np.random.default_rng(seed=seed)
env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)

action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)


# --- Replay Buffer Initialization ---
replay_buffer = deque(maxlen=20000) # Holds past transitions

# --- Model setup (Using LeakyReLU to prevent dead neurons) ---
class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()

        self.liner_relu_stacks = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LeakyReLU(),
            nn.Linear(64, output_dim)        
            )
    
    def forward(self, x):
        return self.liner_relu_stacks(x)

# Double networks to decouple target calculation from active tracking
policy_qNetwork = QNetwork(len(observation_space), len(action_space)).to(device)
target_qNetwork = QNetwork(len(observation_space), len(action_space)).to(device)

target_qNetwork.load_state_dict(policy_qNetwork.state_dict()) # Sync initially

optimizer = optim.Adam(policy_qNetwork.parameters(), lr=alpha)
loss_fn = nn.MSELoss()

# --- Helper Functions ---
def get_state_tensor(state, observation_space):
    one_hot = np.zeros(len(observation_space))
    one_hot[state] = 1
    return torch.tensor(one_hot).unsqueeze(0).float().to(device)

def choose_action(state, epsilon):
    if rng.random() < epsilon:
        return rng.choice(action_space)
    else:
        with torch.no_grad():
            state_tensor = get_state_tensor(state, observation_space)
            q_values = policy_qNetwork(state_tensor)
            return torch.argmax(q_values).item()
        
def generate_qtable(qNetwork, observation_space):
    q_table = np.zeros((len(observation_space), len(action_space)))

    idx = 0
    for action_values in q_table:
        with torch.no_grad():
                state_tensor = get_state_tensor(state=idx, observation_space=observation_space)
                predicted_vals = qNetwork(state_tensor)
                q_table[idx] = np.array(predicted_vals[0])
        idx+=1
    return q_table

def load_DQN_weights():
    weights_path = "DQN_weights.pth"
    if os.path.exists(weights_path):
        policy_qNetwork.load_state_dict(torch.load(weights_path))
        target_qNetwork.load_state_dict(torch.load(weights_path))

def save_DQN_weights():
    weights_path = "DQN_weights.pth"
    torch.save(policy_qNetwork.state_dict(), weights_path)


load_DQN_weights()
q_table = generate_qtable(qNetwork=policy_qNetwork, observation_space=observation_space)
plot_q_table(q_table)

# --- DQN Main Loop ---
for episode in range(num_episodes):
    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False
    episode_reward = 0
    start_time = datetime.now()

    epsilon = max(epsilon * epsilon_decay_rate, epsilon_min)
    td_errors_in_episode = []

    while not (terminated or truncated):
        action = choose_action(state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

        # 1. Store experience inside our replay buffer
        done = terminated or truncated
        replay_buffer.append((state, action, reward, next_state, done))
        state = next_state

        # 2. Train only if we have enough data collected in the buffer
        if len(replay_buffer) > batch_size:
            # Sample a random uncorrelated mini-batch
            mini_batch = random.sample(replay_buffer, batch_size)
            
            states, actions, rewards, next_states, dones = zip(*mini_batch)

            # Process entire batches simultaneously to leverage matrix parallelism
            state_tensors = torch.cat([get_state_tensor(s, observation_space) for s in states])
            next_state_tensors = torch.cat([get_state_tensor(ns, observation_space) for ns in next_states])
            
            reward_tensors = torch.tensor(rewards, dtype=torch.float32).to(device)
            action_tensors = torch.tensor(actions, dtype=torch.long).to(device)
            done_tensors = torch.tensor(dones, dtype=torch.float32).to(device)

            # Get current Q predictions for the chosen actions
            current_q_values = policy_qNetwork(state_tensors)
            predicted_q = current_q_values.gather(1, action_tensors.unsqueeze(1)).squeeze(1)

            # Compute TD Target using the frozen TARGET network
            with torch.no_grad():
                max_next_q = target_qNetwork(next_state_tensors).max(1)[0]
                td_target = reward_tensors + (gamma * max_next_q * (1 - done_tensors))

            # Optimize weights via MSE loss minimization
            loss = loss_fn(predicted_q, td_target)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_qNetwork.parameters(), max_norm=1.0) # Safety clip
            optimizer.step()

            with torch.no_grad():
                td_errors_in_episode.append(torch.abs(td_target - predicted_q).mean().item())

    # 3. Periodically sync Target network parameters with Policy network
    if (episode + 1) % target_update_frequency == 0:
        target_qNetwork.load_state_dict(policy_qNetwork.state_dict())

    # Metrics Logging
    end_time = datetime.now()
    avg_td_error = np.mean(td_errors_in_episode) if td_errors_in_episode else 0.0
    if (episode + 1) % 100 == 0:
        q_table = generate_qtable(qNetwork=policy_qNetwork, observation_space=observation_space)
        plot_q_table(q_table)
        print(f"Episode {episode + 1}/{num_episodes} - Total Reward: {episode_reward}, Epsilon: {epsilon:.3f}, Avg TD Error: {avg_td_error:.4f}")
        save_DQN_weights()

##### Extra: Output to ONNX

In [ ]:
import torch

# 1. Setup paths and input shapes
onnx_model_path = "monteCarlo_qNetwork.onnx"
# Input dimension matches your 48-state one-hot vector size
state_input_dim = 48 

# 2. Put the model into evaluation mode
# This freezes layers like Dropout or BatchNorm so they don't break during export
qNetwork.eval()

# 3. Create a dummy input tensor matching your exact state shape [Batch_Size, State_Dim]
dummy_input = torch.zeros(1, state_input_dim, dtype=torch.float32)

# 4. Export the network graph
print(f"📦 Exporting PyTorch model graph to ONNX format...")
torch.onnx.export(
    qNetwork,                  # The active model instance to export
    dummy_input,               # Tracing input vector
    onnx_model_path,           # Output file path
    export_params=True,        # Store trained parameter weights inside the file
    opset_version=11,          # Standard, highly compatible ONNX operator version
    do_constant_folding=True,  # Optimizes graph by pre-calculating constant math nodes
    input_names=['state'],     # Optional: Give a clear name to your input layer
    output_names=['q_values']  # Optional: Give a clear name to your output layer
)

print(f"✅ ONNX model successfully saved to disk at '{onnx_model_path}'!")